# V5 Selective Z-Mixing — Resilient (Drive Checkpoint)

**Onceki run sonuclari (Phase C, 11/12 tamamlandi):**

| Config | Accepted | Best TM | Best RMSD |
|--------|----------|---------|-----------|
| cut0.05_base0.00_max0.5 | 11/11 (early stop) | 0.6150 | 6.03 |
| cut0.05_base0.00_max0.7 | 19/20 | 0.7798 | 3.17 |
| cut0.05_base0.00_max1.0 | 13/20 | 0.7807 | 3.47 |
| cut0.05_base0.05_max0.5 | 17/20 | 0.7835 | 3.24 |
| cut0.05_base0.05_max0.7 | 9/17 | 0.6782 | 5.19 |
| **cut0.05_base0.05_max1.0** | **6/13** | **0.8420** | **2.63** |
| cut0.10_base0.00_max0.5 | 14/14 (early stop) | 0.7158 | 4.25 |
| cut0.10_base0.00_max0.7 | 20/20 | 0.7812 | 3.20 |
| cut0.10_base0.00_max1.0 | 20/20 | 0.7896 | 3.22 |
| cut0.10_base0.05_max0.5 | 13/20 | 0.7835 | 3.40 |
| **cut0.10_base0.05_max0.7** | **10/17** | **0.8514** | **2.47** |
| cut0.10_base0.05_max1.0 | ??? | KESILDI | - |

**Simdiki gorev:**
1. Eksik Phase C run: `cut0.10_base0.05_max1.0` (1 run)
2. Phase D: Mapping & weight sweep (9 run) — best params: `cutoff=0.10, base=0.05, max=0.7`

**Toplam:** 10 run

## 1. Environment Setup

In [ ]:
import os, sys, shutil

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

if os.path.exists('/content/ANM-openfold3'):
    shutil.rmtree('/content/ANM-openfold3')
!git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
    !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
    !pip install -e /content/ANM-openfold3/openfold3-repo -q
else:
    try:
        import openfold3
    except ImportError:
        !pip install -e /content/ANM-openfold3/openfold3-repo -q

!pip install -q biopython matplotlib numpy torch pandas

sys.path.insert(0, '/content/ANM-openfold3')
sys.path.insert(0, '/content/ANM-openfold3/openfold3-repo')

os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)

from pathlib import Path
from openfold3.entry_points.parameters import (
    download_model_parameters,
    get_default_checkpoint_dir,
    DEFAULT_CHECKPOINT_NAME,
)
_param_dir = get_default_checkpoint_dir()
_param_dir.mkdir(parents=True, exist_ok=True)
download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)

print('Environment setup complete.')

## 2. Config + Checkpoint Utils

In [ ]:
import json
from datetime import datetime

# ════════════════════════════════════════════════════
#  DRIVE CHECKPOINT CONFIG
# ════════════════════════════════════════════════════
DRIVE_SAVE_DIR = '/content/drive/MyDrive/ANM-openfold3/search_v5'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)


def checkpoint_path(phase_name: str) -> str:
    return os.path.join(DRIVE_SAVE_DIR, f'checkpoint_{phase_name}.json')


def save_checkpoint(phase_name: str, results: list, derived_params: dict = None):
    """Faz sonuclarini Drive'a kaydet."""
    # Temizle: tensor/numpy -> list
    clean_results = []
    for r in results:
        r_clean = {}
        for k, v in r.items():
            if k == 'mask_snapshots':
                # numpy array -> list
                r_clean[k] = [
                    {'step': s['step'], 'mask': s['mask'].tolist()}
                    for s in v
                ]
            else:
                r_clean[k] = v
        clean_results.append(r_clean)

    data = {
        'phase': phase_name,
        'completed_at': datetime.now().isoformat(),
        'n_runs': len(results),
        'results': clean_results,
        'derived_params': derived_params or {},
    }
    path = checkpoint_path(phase_name)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2, default=str)
    print(f'  [SAVED] {phase_name} -> {path} ({len(results)} runs)')


def load_checkpoint(phase_name: str) -> dict | None:
    """Drive'dan checkpoint yukle. Yoksa None dondur."""
    path = checkpoint_path(phase_name)
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f'  [RESUME] {phase_name} Drive\'dan yuklendi ({data["n_runs"]} runs, {data["completed_at"]})')
        return data
    return None


def save_partial(phase_name: str, results_so_far: list):
    """Faz icinde her run sonunda partial kayit (kesintiye karsi)."""
    path = os.path.join(DRIVE_SAVE_DIR, f'partial_{phase_name}.json')
    clean = []
    for r in results_so_far:
        r_clean = {k: v for k, v in r.items() if k != 'mask_snapshots'}
        clean.append(r_clean)
    with open(path, 'w') as f:
        json.dump({'phase': phase_name, 'partial': True, 'n_runs': len(clean),
                   'saved_at': datetime.now().isoformat(), 'results': clean},
                  f, indent=2, default=str)


# ════════════════════════════════════════════════════
#  PIPELINE CONFIG (V5 — V4'ten sabit parametreler)
# ════════════════════════════════════════════════════

# -- PDB --
INITIAL_PDB = '1AKE'
TARGET_PDB  = '4AKE'
CHAIN_ID    = 'A'

# -- Pipeline (V4'ten sabit) --
N_STEPS              = 20
COMBINATION_STRATEGY = 'autostop'
Z_MIXING_ALPHA       = 0.7
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = 'plus'

# -- OF3 --
USE_MSA_SERVER        = True
USE_TEMPLATES         = False
NUM_ROLLOUT_STEPS     = None
NUM_DIFFUSION_SAMPLES = 1

# -- Confidence cutoffs --
ALPHA_DECAY               = 0.8
MAX_STALL                 = 8
CONFIDENCE_PTM_CUTOFF     = 0.30
CONFIDENCE_PLDDT_CUTOFF   = 65.0
CONFIDENCE_RANKING_CUTOFF = 0.45
CONFIDENCE_RG_MAX         = 2.5
CONFIDENCE_RG_MIN         = 0.3
CONFIDENCE_CLASH_REJECT   = True
CONFIDENCE_RMSD_INIT_MAX  = 10.0

# -- Drift korumalari --
ENABLE_BEST_ROLLBACK  = True
BEST_ROLLBACK_TM_DROP = 0.40
ENABLE_ADAPTIVE_STOP  = True
ADAPTIVE_STOP_WINDOW  = 3

# -- Fallback --
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True
AUTOSTOP_FALLBACK_LEVELS  = (0, 1, 4, 9)

# -- Autostop --
AUTOSTOP_V_MAGNITUDE = 1.0
AUTOSTOP_N_STEPS     = 5000
AUTOSTOP_BACK_OFF    = 2

# -- Converter --
DRIVE_DIR = '/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa'
CHECKPOINT_SELECTION = 'best_bf_r'

# ════════════════════════════════════════════════════
#  SELECTIVE MIXING SWEEP PARAMETRELERI
# ════════════════════════════════════════════════════
SELECTIVE_CHANGE_CUTOFFS = [0.05, 0.1, 0.15, 0.2, 0.3]
SELECTIVE_ALPHA_BASES    = [0.0, 0.05]
SELECTIVE_ALPHA_MAXES    = [0.5, 0.7, 1.0]
SELECTIVE_MAPPINGS       = ['linear', 'sigmoid', 'step']
SELECTIVE_W_PAIRS        = [(0.3, 0.7), (0.5, 0.5), (0.7, 0.3)]

print('Config ready.')
print(f'  Drive save dir: {DRIVE_SAVE_DIR}')
print(f'  Confidence: pTM>={CONFIDENCE_PTM_CUTOFF} ranking>={CONFIDENCE_RANKING_CUTOFF} pLDDT>={CONFIDENCE_PLDDT_CUTOFF}')

## 3. Converter + PDB + OF3

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json as _json
import numpy as np
import torch
import time
import copy
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score
from src.composite_confidence import (
    CompositeWeights, compute_composite, composite_score_from_step,
    WEIGHT_PRESETS, THRESHOLD_PRESETS,
    normalize_ptm, normalize_plddt, normalize_pae, normalize_rg, normalize_contact_recon,
)
from src.selective_mixing import compute_change_score, compute_alpha_mask, selective_blend_z

# -- Checkpoint --
history_path = os.path.join(DRIVE_DIR, 'history.json')
best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
    with open(history_path) as f:
        history = _json.load(f)
    best_bf_r = -1
    best_epoch = -1
    for entry in history:
        bf_r = entry.get('val_bf_pearson', 0)
        if bf_r > best_bf_r:
            best_bf_r = bf_r
            best_epoch = entry['epoch']
    epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
    CHECKPOINT_PATH = epoch_ckpt if os.path.exists(epoch_ckpt) else best_model_path
    print(f'Best bf_r: {best_bf_r:.4f} at epoch {best_epoch}')
else:
    CHECKPOINT_PATH = best_model_path

converter = PairContactConverter(CHECKPOINT_PATH)
print(f'Converter loaded from {CHECKPOINT_PATH}')

In [ ]:
# -- PDB --
from Bio.PDB import PDBParser, PDBList
from Bio.SeqUtils import seq1

def fetch_ca(pdb_id, chain_id):
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb_cache', file_format='pdb')
    structure = parser.get_structure(pdb_id, pdb_file)
    chain = [c for c in structure[0].get_chains() if c.id == chain_id][0]
    residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    coords = torch.tensor([r['CA'].get_vector().get_array() for r in residues], dtype=torch.float32)
    sequence = ''.join(seq1(r.get_resname()) for r in residues)
    return coords, sequence

initial_ca, sequence = fetch_ca(INITIAL_PDB, CHAIN_ID)
target_ca, _ = fetch_ca(TARGET_PDB, CHAIN_ID)
N = initial_ca.shape[0]
print(f'Initial: {INITIAL_PDB} chain {CHAIN_ID}, N={N}')
print(f'Target:  {TARGET_PDB} chain {CHAIN_ID}, N={target_ca.shape[0]}')
print(f'RMSD to target: {compute_rmsd(initial_ca, target_ca):.2f} A')
print(f'TM-score: {tm_score(initial_ca, target_ca):.4f}')

# -- OF3 --
_query_dir = '/content/of3_queries'
os.makedirs(_query_dir, exist_ok=True)
_query = {
    'queries': {
        INITIAL_PDB: {
            'chains': [{
                'molecule_type': 'protein',
                'chain_ids': [CHAIN_ID],
                'sequence': sequence,
            }]
        }
    }
}
_query_path = os.path.join(_query_dir, f'{INITIAL_PDB}.json')
with open(_query_path, 'w') as f:
    _json.dump(_query, f, indent=2)

from src.of3_diffusion import load_of3_diffusion
diffusion_fn, zij_trunk = load_of3_diffusion(
    query_json=_query_path,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
    num_rollout_steps=NUM_ROLLOUT_STEPS,
    num_samples=NUM_DIFFUSION_SAMPLES,
)
print(f'zij_trunk shape: {zij_trunk.shape}')

from src.autostop_adapter import StructureContext
_pdb_file = PDBList().retrieve_pdb_file(INITIAL_PDB, pdir='/tmp/pdb_cache', file_format='pdb')
structure_ctx = StructureContext.from_pdb(_pdb_file, chain_id=CHAIN_ID)
print(f'StructureContext: N={structure_ctx.struct.N}')
print('OF3 ready.')

## 4. V5 Runner

In [ ]:
# ════════════════════════════════════════════════════
#  V5 RUNNER (V4 safeguards + selective mixing)
# ════════════════════════════════════════════════════

def make_v5_config(
    n_steps=N_STEPS,
    alpha=Z_MIXING_ALPHA,
    alpha_decay=ALPHA_DECAY,
    max_stall=MAX_STALL,
    selective_mixing=False,
    selective_params=None,
):
    sel = selective_params or {}
    return ModeDriveConfig(
        n_steps=n_steps,
        combination_strategy=COMBINATION_STRATEGY,
        z_mixing_alpha=alpha,
        n_anm_modes=N_ANM_MODES,
        n_combinations=N_COMBINATIONS,
        max_combo_size=MAX_COMBO_SIZE,
        df=DF, df_min=DF_MIN, df_max=DF_MAX,
        anm_cutoff=ANM_CUTOFF,
        contact_r_cut=CONTACT_R_CUT,
        contact_tau=CONTACT_TAU,
        z_direction=Z_DIRECTION,
        num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
        confidence_ptm_cutoff=CONFIDENCE_PTM_CUTOFF,
        confidence_plddt_cutoff=CONFIDENCE_PLDDT_CUTOFF,
        confidence_ranking_cutoff=CONFIDENCE_RANKING_CUTOFF,
        confidence_rg_max=CONFIDENCE_RG_MAX,
        confidence_rg_min=CONFIDENCE_RG_MIN,
        confidence_clash_reject=CONFIDENCE_CLASH_REJECT,
        confidence_rmsd_init_max=CONFIDENCE_RMSD_INIT_MAX,
        enable_confidence_fallback=ENABLE_FALLBACK,
        fallback_extended_enabled=FALLBACK_EXTENDED_ENABLED,
        autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
        autostop_n_steps=AUTOSTOP_N_STEPS,
        autostop_back_off=AUTOSTOP_BACK_OFF,
        autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
        autostop_chain_id=CHAIN_ID,
        rejected_alpha_decay=alpha_decay,
        max_consecutive_rejected=max_stall,
        confidence_warmup_steps=0,
        enable_best_rollback=ENABLE_BEST_ROLLBACK,
        best_rollback_tm_drop=BEST_ROLLBACK_TM_DROP,
        enable_adaptive_stop=ENABLE_ADAPTIVE_STOP,
        adaptive_stop_window=ADAPTIVE_STOP_WINDOW,
        # V5 selective mixing
        selective_mixing=selective_mixing,
        selective_w_connectivity=sel.get('w_connectivity', 0.5),
        selective_w_distance=sel.get('w_distance', 0.5),
        selective_change_cutoff=sel.get('change_cutoff', 0.1),
        selective_alpha_base=sel.get('alpha_base', 0.0),
        selective_alpha_max=sel.get('alpha_max', 1.0),
        selective_mapping=sel.get('mapping', 'linear'),
        selective_distance_mode=sel.get('distance_mode', 'max'),
    )


def run_v5(
    weights: CompositeWeights,
    threshold: float,
    label: str = '',
    n_steps: int = N_STEPS,
    alpha: float = Z_MIXING_ALPHA,
    alpha_decay: float = ALPHA_DECAY,
    max_stall: int = MAX_STALL,
    early_abort_steps: int = 5,
    selective_mixing: bool = False,
    selective_params: dict | None = None,
    verbose: bool = True,
) -> dict:
    """V5 runner with composite score, drift protection, selective mixing."""
    cfg = make_v5_config(
        n_steps=n_steps, alpha=alpha,
        alpha_decay=alpha_decay, max_stall=max_stall,
        selective_mixing=selective_mixing,
        selective_params=selective_params,
    )
    pipe = ModeDrivePipeline(
        converter=converter, config=cfg,
        diffusion_fn=diffusion_fn,
        structure_ctx=structure_ctx,
    )

    sel_tag = 'SELECTIVE' if selective_mixing else 'UNIFORM'
    sel_info = ''
    if selective_mixing and selective_params:
        sp = selective_params
        sel_info = (
            f'  selective: cutoff={sp.get("change_cutoff", 0.1)} '
            f'base={sp.get("alpha_base", 0.0)} '
            f'max={sp.get("alpha_max", 1.0)} '
            f'map={sp.get("mapping", "linear")} '
            f'w=({sp.get("w_connectivity", 0.5)},{sp.get("w_distance", 0.5)})'
        )

    print(f'\n{"="*70}')
    print(f'  {label} [{sel_tag}]')
    if sel_info:
        print(sel_info)
    print(f'{"="*70}')

    coords_ca = initial_ca.clone()
    z_current = zij_trunk.clone()
    orig_alpha = alpha
    current_alpha = alpha
    consecutive_rejected = 0

    coords_best = initial_ca.clone()
    z_best = zij_trunk.clone()
    tm_best = 0.0
    best_step_idx = -1
    rollback_count = 0
    accepted_tm_history = []

    step_metrics = []
    mask_snapshots = []
    t0 = time.time()
    stopped_early = False

    for step_idx in range(n_steps):
        cfg.z_mixing_alpha = current_alpha
        sr = pipe.step_with_fallback(
            coords_ca, initial_ca, z_current,
            prev_rmsd=0.0,
            target_coords=target_ca,
            step_idx=step_idx,
        )

        if sr.alpha_mask_snapshot is not None:
            mask_snapshots.append({
                'step': step_idx + 1,
                'mask': sr.alpha_mask_snapshot.numpy(),
            })

        comp_score, comp_detail = composite_score_from_step(sr, weights)

        hard_reject = False
        reject_reason = ''
        if sr.rg_ratio is not None and sr.rg_ratio > CONFIDENCE_RG_MAX:
            hard_reject = True
            reject_reason = f'Rg={sr.rg_ratio:.1f}>{CONFIDENCE_RG_MAX}'
        if sr.has_clash and CONFIDENCE_CLASH_REJECT:
            hard_reject = True
            reject_reason = 'clash'
        if sr.rmsd > CONFIDENCE_RMSD_INIT_MAX:
            hard_reject = True
            reject_reason = f'rmsd_init={sr.rmsd:.1f}>{CONFIDENCE_RMSD_INIT_MAX}'

        if hard_reject:
            accepted = False
        else:
            accepted = comp_score >= threshold
            if not accepted:
                reject_reason = f'comp={comp_score:.3f}<{threshold}'

        rmsd_tgt = compute_rmsd(sr.new_ca, target_ca)
        tm_tgt = tm_score(sr.new_ca, target_ca)

        m = {
            'step': step_idx + 1,
            'accepted': accepted,
            'comp_score': comp_score,
            'rmsd_init': sr.rmsd,
            'rmsd_tgt': rmsd_tgt,
            'tm_tgt': tm_tgt,
            'ptm': sr.ptm,
            'plddt_mean': float(sr.plddt.mean()) if sr.plddt is not None else None,
            'ranking': sr.ranking_score,
            'mean_pae': sr.mean_pae,
            'rg_ratio': sr.rg_ratio,
            'has_clash': sr.has_clash,
            'fallback_level': sr.fallback_level,
            'alpha_used': current_alpha,
            'reject_reason': reject_reason,
            'rollback': False,
            'change_score_mean': sr.change_score_mean,
            'change_score_max': sr.change_score_max,
            'n_active_pairs': sr.n_active_pairs,
            'alpha_mask_mean': sr.alpha_mask_mean,
        }

        if verbose:
            tag = 'ACC' if accepted else 'REJ'
            sel_diag = ''
            if selective_mixing and sr.change_score_mean is not None:
                sel_diag = f' cs={sr.change_score_mean:.3f} np={sr.n_active_pairs}'
            extra = f' {reject_reason}' if not accepted else ''
            print(
                f'  [{step_idx+1:>2}/{n_steps}] {tag}  '
                f'comp={comp_score:.3f} TM={tm_tgt:.3f} '
                f'RMSD={rmsd_tgt:.2f} a={current_alpha:.3f}'
                f'{sel_diag}{extra}'
            )

        if accepted:
            coords_ca = sr.new_ca
            z_current = sr.z_modified
            consecutive_rejected = 0
            current_alpha = orig_alpha

            if tm_tgt > tm_best:
                tm_best = tm_tgt
                coords_best = sr.new_ca.clone()
                z_best = sr.z_modified.clone()
                best_step_idx = step_idx
            elif tm_best > 0 and tm_tgt < tm_best * (1.0 - BEST_ROLLBACK_TM_DROP):
                if verbose:
                    print(f'  >> ROLLBACK to step {best_step_idx+1}')
                coords_ca = coords_best.clone()
                z_current = z_best.clone()
                rollback_count += 1
                m['rollback'] = True

            accepted_tm_history.append(tm_tgt)
            if len(accepted_tm_history) >= ADAPTIVE_STOP_WINDOW + 1:
                recent = accepted_tm_history[-(ADAPTIVE_STOP_WINDOW + 1):]
                if all(recent[i] > recent[i+1] for i in range(len(recent)-1)):
                    if verbose:
                        print(f'  >> EARLY STOP: {ADAPTIVE_STOP_WINDOW} ardisik TM dusus')
                    stopped_early = True
                    step_metrics.append(m)
                    break
        else:
            consecutive_rejected += 1
            if alpha_decay < 1.0:
                current_alpha = max(0.02, current_alpha * alpha_decay)
            if max_stall > 0 and consecutive_rejected >= max_stall:
                if verbose:
                    print(f'  STALL: {consecutive_rejected} consecutive rejected')
                break

        step_metrics.append(m)

        if step_idx + 1 == early_abort_steps:
            n_acc = sum(1 for x in step_metrics if x['accepted'])
            if n_acc == 0:
                if verbose:
                    print(f'  EARLY ABORT: 0/{early_abort_steps} accepted')
                break

    wall = time.time() - t0
    total_steps = len(step_metrics)
    n_accepted = sum(1 for x in step_metrics if x['accepted'])

    accepted_metrics = [x for x in step_metrics if x['accepted']]
    if accepted_metrics:
        best_step = max(accepted_metrics, key=lambda x: x['tm_tgt'])
        best_tm = best_step['tm_tgt']
        best_rmsd = best_step['rmsd_tgt']
    else:
        best_tm = tm_score(initial_ca, target_ca)
        best_rmsd = compute_rmsd(initial_ca, target_ca)

    result = {
        'label': label,
        'threshold': threshold,
        'alpha': alpha,
        'alpha_decay': alpha_decay,
        'selective_mixing': selective_mixing,
        'selective_params': selective_params or {},
        'total_steps': total_steps,
        'accepted': n_accepted,
        'rejected': total_steps - n_accepted,
        'accept_rate': n_accepted / max(1, total_steps),
        'best_tm': best_tm,
        'best_rmsd': best_rmsd,
        'rollback_count': rollback_count,
        'stopped_early': stopped_early,
        'wall_s': wall,
        'step_metrics': step_metrics,
        'mask_snapshots': mask_snapshots,
    }

    print(f'  => {n_accepted}/{total_steps} acc  '
          f'best_TM={best_tm:.4f}  best_RMSD={best_rmsd:.2f}  wall={wall:.0f}s')

    # Inline heatmaps (selective only)
    if selective_mixing and mask_snapshots and verbose:
        n_show = min(len(mask_snapshots), 4)
        indices = np.linspace(0, len(mask_snapshots)-1, n_show, dtype=int)
        fig, axes = plt.subplots(1, n_show, figsize=(3.5*n_show, 3))
        if n_show == 1:
            axes = [axes]
        for ax, idx in zip(axes, indices):
            snap = mask_snapshots[idx]
            ax.imshow(snap['mask'], cmap='hot', vmin=0, vmax=1, aspect='equal')
            ax.set_title(f'Step {snap["step"]}', fontsize=9)
        fig.suptitle(f'{label}', fontsize=10)
        plt.tight_layout()
        plt.show()

    return result


# V4 best config
V4_BEST_WEIGHTS = WEIGHT_PRESETS['A_ptm_heavy']
V4_BEST_THRESHOLD = 0.50
V4_BEST_ALPHA = 0.7
V4_BEST_DECAY = 0.8

print('V5 Runner ready.')

## 5. Phase C Sonuclari (onceki run'dan) + Eksik Run

11/12 run onceden tamamlandi. Sadece eksik olan `C_cut0.10_base0.05_max1.0` calistirilacak.
Best so far: **cutoff=0.10, base=0.05, max=0.7 → TM=0.8514**

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE C: Onceki sonuclar + eksik run
# ════════════════════════════════════════════════════

# Onceki run'dan hard-coded sonuclar (step_metrics yok, sadece ozet)
PRIOR_RESULTS_C = [
    {'label': 'C_cut0.05_base0.00_max0.5', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.05, 'alpha_base': 0.0, 'alpha_max': 0.5, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 11, 'accepted': 11, 'rejected': 0, 'accept_rate': 1.0, 'best_tm': 0.6150, 'best_rmsd': 6.03, 'rollback_count': 0, 'stopped_early': True, 'wall_s': 457, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.05_base0.00_max0.7', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.05, 'alpha_base': 0.0, 'alpha_max': 0.7, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 20, 'accepted': 19, 'rejected': 1, 'accept_rate': 0.95, 'best_tm': 0.7798, 'best_rmsd': 3.17, 'rollback_count': 0, 'stopped_early': False, 'wall_s': 2236, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.05_base0.00_max1.0', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.05, 'alpha_base': 0.0, 'alpha_max': 1.0, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 20, 'accepted': 13, 'rejected': 7, 'accept_rate': 0.65, 'best_tm': 0.7807, 'best_rmsd': 3.47, 'rollback_count': 0, 'stopped_early': False, 'wall_s': 2485, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.05_base0.05_max0.5', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.05, 'alpha_base': 0.05, 'alpha_max': 0.5, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 20, 'accepted': 17, 'rejected': 3, 'accept_rate': 0.85, 'best_tm': 0.7835, 'best_rmsd': 3.24, 'rollback_count': 2, 'stopped_early': False, 'wall_s': 2719, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.05_base0.05_max0.7', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.05, 'alpha_base': 0.05, 'alpha_max': 0.7, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 17, 'accepted': 9, 'rejected': 8, 'accept_rate': 0.53, 'best_tm': 0.6782, 'best_rmsd': 5.19, 'rollback_count': 0, 'stopped_early': False, 'wall_s': 2764, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.05_base0.05_max1.0', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.05, 'alpha_base': 0.05, 'alpha_max': 1.0, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 13, 'accepted': 6, 'rejected': 7, 'accept_rate': 0.46, 'best_tm': 0.8420, 'best_rmsd': 2.63, 'rollback_count': 0, 'stopped_early': False, 'wall_s': 1997, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.10_base0.00_max0.5', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.10, 'alpha_base': 0.0, 'alpha_max': 0.5, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 14, 'accepted': 14, 'rejected': 0, 'accept_rate': 1.0, 'best_tm': 0.7158, 'best_rmsd': 4.25, 'rollback_count': 0, 'stopped_early': True, 'wall_s': 555, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.10_base0.00_max0.7', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.10, 'alpha_base': 0.0, 'alpha_max': 0.7, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 20, 'accepted': 20, 'rejected': 0, 'accept_rate': 1.0, 'best_tm': 0.7812, 'best_rmsd': 3.20, 'rollback_count': 0, 'stopped_early': False, 'wall_s': 1730, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.10_base0.00_max1.0', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.10, 'alpha_base': 0.0, 'alpha_max': 1.0, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 20, 'accepted': 20, 'rejected': 0, 'accept_rate': 1.0, 'best_tm': 0.7896, 'best_rmsd': 3.22, 'rollback_count': 0, 'stopped_early': False, 'wall_s': 2350, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.10_base0.05_max0.5', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.10, 'alpha_base': 0.05, 'alpha_max': 0.5, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 20, 'accepted': 13, 'rejected': 7, 'accept_rate': 0.65, 'best_tm': 0.7835, 'best_rmsd': 3.40, 'rollback_count': 0, 'stopped_early': False, 'wall_s': 2812, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
    {'label': 'C_cut0.10_base0.05_max0.7', 'selective_mixing': True, 'selective_params': {'change_cutoff': 0.10, 'alpha_base': 0.05, 'alpha_max': 0.7, 'mapping': 'linear', 'w_connectivity': 0.5, 'w_distance': 0.5}, 'total_steps': 17, 'accepted': 10, 'rejected': 7, 'accept_rate': 0.59, 'best_tm': 0.8514, 'best_rmsd': 2.47, 'rollback_count': 0, 'stopped_early': False, 'wall_s': 2633, 'step_metrics': [], 'mask_snapshots': [], 'threshold': 0.50, 'alpha': 0.7, 'alpha_decay': 0.8},
]

# Eksik run: C_cut0.10_base0.05_max1.0
ckpt_c = load_checkpoint('phase_c')

if ckpt_c is not None:
    RESULTS_C = ckpt_c['results']
    print(f'  Phase C Drive\'dan yuklendi ({len(RESULTS_C)} run)')
else:
    RESULTS_C = list(PRIOR_RESULTS_C)  # onceki 11 run

    # Eksik run'i calistir
    print('Eksik run baslatiliyor: C_cut0.10_base0.05_max1.0')
    r_missing = run_v5(
        weights=V4_BEST_WEIGHTS,
        threshold=V4_BEST_THRESHOLD,
        label='C_cut0.10_base0.05_max1.0',
        alpha=V4_BEST_ALPHA,
        alpha_decay=V4_BEST_DECAY,
        selective_mixing=True,
        selective_params={
            'change_cutoff': 0.10,
            'alpha_base': 0.05,
            'alpha_max': 1.0,
            'mapping': 'linear',
            'w_connectivity': 0.5,
            'w_distance': 0.5,
        },
    )
    RESULTS_C.append(r_missing)

    # Checkpoint kaydet
    best_c = max(RESULTS_C, key=lambda r: (r['best_tm'], r['accepted']))
    BEST_CUTOFF = best_c['selective_params']['change_cutoff']
    BEST_ALPHA_BASE = best_c['selective_params']['alpha_base']
    BEST_ALPHA_MAX = best_c['selective_params']['alpha_max']
    save_checkpoint('phase_c', RESULTS_C, {
        'BEST_CUTOFF': BEST_CUTOFF,
        'BEST_ALPHA_BASE': BEST_ALPHA_BASE,
        'BEST_ALPHA_MAX': BEST_ALPHA_MAX,
    })

# Derive best params (regardless of source)
best_c = max(RESULTS_C, key=lambda r: (r['best_tm'], r['accepted']))
BEST_CUTOFF = best_c['selective_params']['change_cutoff']
BEST_ALPHA_BASE = best_c['selective_params']['alpha_base']
BEST_ALPHA_MAX = best_c['selective_params']['alpha_max']

# Sonuc tablosu
rows = []
for r in RESULTS_C:
    sp = r.get('selective_params', {})
    rows.append({
        'label': r['label'],
        'cutoff': sp.get('change_cutoff', '-'),
        'base': sp.get('alpha_base', '-'),
        'max': sp.get('alpha_max', '-'),
        'accepted': r['accepted'],
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
    })
print('\nPHASE C SONUC (12 run):')
print(pd.DataFrame(rows).to_string(index=False))
print(f'\nBest: cutoff={BEST_CUTOFF} base={BEST_ALPHA_BASE} max={BEST_ALPHA_MAX} → TM={best_c["best_tm"]:.4f}')

## 6. Phase D: Mapping & Weight Sweep (9 run)

Phase C'nin en iyi parametreleri ile:
- Mapping: linear, sigmoid, step
- Weight: (w_c, w_d) = (0.3,0.7), (0.5,0.5), (0.7,0.3)

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE D: MAPPING & WEIGHT SWEEP (9 run)
# ════════════════════════════════════════════════════

ckpt_d = load_checkpoint('phase_d')

if ckpt_d is not None:
    RESULTS_D = ckpt_d['results']
    print(f'  Phase D atlanıyor ({len(RESULTS_D)} run)')
else:
    RESULTS_D = []

    for mapping in SELECTIVE_MAPPINGS:
        for w_c, w_d in SELECTIVE_W_PAIRS:
            label = f'D_{mapping}_wc{w_c:.1f}_wd{w_d:.1f}'
            r = run_v5(
                weights=V4_BEST_WEIGHTS,
                threshold=V4_BEST_THRESHOLD,
                label=label,
                alpha=V4_BEST_ALPHA,
                alpha_decay=V4_BEST_DECAY,
                selective_mixing=True,
                selective_params={
                    'change_cutoff': BEST_CUTOFF,
                    'alpha_base': BEST_ALPHA_BASE,
                    'alpha_max': BEST_ALPHA_MAX,
                    'mapping': mapping,
                    'w_connectivity': w_c,
                    'w_distance': w_d,
                },
            )
            RESULTS_D.append(r)
            save_partial('phase_d', RESULTS_D)

    save_checkpoint('phase_d', RESULTS_D)

# Sonuc tablosu
rows = []
for r in RESULTS_D:
    sp = r.get('selective_params', {})
    rows.append({
        'label': r['label'],
        'mapping': sp.get('mapping', '-'),
        'w_c': sp.get('w_connectivity', '-'),
        'w_d': sp.get('w_distance', '-'),
        'accepted': r['accepted'],
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
    })
print('\nPHASE D SONUC:')
print(pd.DataFrame(rows).to_string(index=False))

best_d = max(RESULTS_D, key=lambda r: (r['best_tm'], r['accepted']))
print(f'Best Phase D: {best_d["label"]} -- TM={best_d["best_tm"]:.4f}')

## 7. Analysis & Final Summary

Tum fazlari birlestir, en iyi sonuclari goster, grafikler ciz, Drive'a kaydet.

In [ ]:
# ════════════════════════════════════════════════════
#  ANALYSIS — GENEL SONUC
# ════════════════════════════════════════════════════

ALL_RESULTS = {
    'phase_c': RESULTS_C,
    'phase_d': RESULTS_D,
}

# ── Genel tablo ──
all_rows = []
for phase_name, results in ALL_RESULTS.items():
    for r in results:
        sp = r.get('selective_params', {})
        all_rows.append({
            'phase': phase_name,
            'label': r['label'],
            'selective': r.get('selective_mixing', True),
            'best_TM': r['best_tm'],
            'best_RMSD': r['best_rmsd'],
            'accepted': r['accepted'],
            'cutoff': sp.get('change_cutoff', ''),
            'base': sp.get('alpha_base', ''),
            'max': sp.get('alpha_max', ''),
            'mapping': sp.get('mapping', ''),
        })
df_all = pd.DataFrame(all_rows).sort_values('best_TM', ascending=False)
print('='*90)
print('GENEL SONUC TABLOSU (sorted by TM)')
print('='*90)
print(df_all.to_string(index=False))

# En iyi genel
all_flat = []
for results in ALL_RESULTS.values():
    all_flat.extend(results)
best_overall = max(all_flat, key=lambda r: (r['best_tm'], r['accepted']))
print(f'\nEN IYI: {best_overall["label"]} — TM={best_overall["best_tm"]:.4f} RMSD={best_overall["best_rmsd"]:.2f}')

if best_overall.get('selective_mixing', True):
    sp = best_overall.get('selective_params', {})
    print(f'\nEn iyi selective params:')
    print(f'  change_cutoff  = {sp.get("change_cutoff", "-")}')
    print(f'  alpha_base     = {sp.get("alpha_base", "-")}')
    print(f'  alpha_max      = {sp.get("alpha_max", "-")}')
    print(f'  mapping        = {sp.get("mapping", "-")}')
    print(f'  w_connectivity = {sp.get("w_connectivity", "-")}')
    print(f'  w_distance     = {sp.get("w_distance", "-")}')

In [ ]:
# ── Grafikler ──

# 1. TM-score bar chart
fig, ax = plt.subplots(figsize=(12, max(6, len(all_flat)*0.3)))
labels_plot = [r['label'][-30:] for r in all_flat]
tms = [r['best_tm'] for r in all_flat]
colors = ['#3498db' for _ in all_flat]
ax.barh(labels_plot, tms, color=colors)
ax.set_xlabel('Best TM-score')
ax.set_title('V5 Selective Mixing — All Runs')
plt.tight_layout()
plt.show()

# 2. En iyi run step trajectory (sadece step_metrics varsa)
if best_overall.get('step_metrics'):
    metrics = best_overall['step_metrics']
    steps = [m['step'] for m in metrics]
    tm_vals = [m['tm_tgt'] for m in metrics]
    accepted_flags = [m['accepted'] for m in metrics]

    fig, ax = plt.subplots(figsize=(12, 4))
    for s, t, a in zip(steps, tm_vals, accepted_flags):
        color = '#2ecc71' if a else '#e74c3c'
        ax.scatter(s, t, color=color, s=80, zorder=5, edgecolors='black', linewidths=0.5)
    ax.plot(steps, tm_vals, 'k-', alpha=0.3)
    ax.set_xlabel('Step')
    ax.set_ylabel('TM-score vs target')
    ax.set_title(f'Best run: {best_overall["label"]}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f'Not: {best_overall["label"]} onceki run\'dan — step_metrics kayitli degil.')

In [ ]:
# ════════════════════════════════════════════════════
#  FINAL SAVE TO DRIVE
# ════════════════════════════════════════════════════

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Summary JSON (lightweight — no step_metrics, no mask_snapshots)
summary_rows = []
for phase_name, results in ALL_RESULTS.items():
    for r in results:
        sp = r.get('selective_params', {})
        summary_rows.append({
            'phase': phase_name,
            'label': r['label'],
            'selective_mixing': r.get('selective_mixing', True),
            'change_cutoff': sp.get('change_cutoff', ''),
            'alpha_base': sp.get('alpha_base', ''),
            'alpha_max': sp.get('alpha_max', ''),
            'mapping': sp.get('mapping', ''),
            'w_connectivity': sp.get('w_connectivity', ''),
            'w_distance': sp.get('w_distance', ''),
            'best_tm': r['best_tm'],
            'best_rmsd': r['best_rmsd'],
            'accepted': r['accepted'],
            'total_steps': r['total_steps'],
            'accept_rate': r.get('accept_rate', 0),
            'rollbacks': r.get('rollback_count', 0),
            'early_stop': r.get('stopped_early', False),
            'wall_s': r.get('wall_s', 0),
        })

summary_path = os.path.join(DRIVE_SAVE_DIR, f'summary_final_{timestamp}.json')
with open(summary_path, 'w') as f:
    json.dump({
        'completed_at': datetime.now().isoformat(),
        'total_runs': len(summary_rows),
        'best_label': best_overall['label'],
        'best_tm': best_overall['best_tm'],
        'best_rmsd': best_overall['best_rmsd'],
        'best_params': best_overall.get('selective_params', {}),
        'results': summary_rows,
    }, f, indent=2, default=str)
print(f'Summary JSON: {summary_path}')

# Summary CSV
csv_path = os.path.join(DRIVE_SAVE_DIR, f'summary_final_{timestamp}.csv')
pd.DataFrame(summary_rows).to_csv(csv_path, index=False)
print(f'Summary CSV:  {csv_path}')

print(f'\nTamamlandi! {len(summary_rows)} run sonucu Drive\'a kaydedildi.')
print(f'Drive dir: {DRIVE_SAVE_DIR}')